# 03 · Fine-tuning and extending a deployed model (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/03_finetune_extend.py`.

**Traces to** `notes/ioai-task-patterns.md` P4 · T26-H-NIGHT (grow the head, replay, KD, freeze/LoRA),
T25-CHICKEN (partial state-dict loading). Section 4 builds LoRA from scratch.

Total suggested time: **55 min**.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

tr = datasets.FashionMNIST("data", train=True, download=True); te = datasets.FashionMNIST("data", train=False, download=True)
X_all, y_all = tr.data.unsqueeze(1).float() / 255, tr.targets
X_test, y_test = te.data.unsqueeze(1).float() / 255, te.targets

class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.head = nn.Linear(128, n_classes)
    def forward(self, x):
        return self.head(self.features(x))

deployed = Net(5).to(device)
_opt = torch.optim.Adam(deployed.parameters(), 1e-3)
_m = y_all < 5
for _ in range(2):
    for xb, yb in DataLoader(TensorDataset(X_all[_m], y_all[_m]), batch_size=64, shuffle=True):
        loss = F.cross_entropy(deployed(xb.to(device)), yb.to(device)); _opt.zero_grad(); loss.backward(); _opt.step()
deployed.eval()
g = torch.Generator().manual_seed(0)
pick = lambda cs, k: torch.cat([torch.nonzero(y_all == c).flatten()[torch.randperm(int((y_all == c).sum()), generator=g)[:k]] for c in cs])
i_old, i_new = pick(range(5), 20), pick(range(5, 10), 200)
X_old, y_old, X_new, y_new = X_all[i_old], y_all[i_old], X_all[i_new], y_all[i_new]

@torch.no_grad()
def night_score(model):
    model.eval()
    pred = torch.cat([model(X_test[i:i + 2000].to(device)).argmax(1).cpu() for i in range(0, 10000, 2000)])
    o, n = y_test < 5, y_test >= 5
    return 0.5 * (pred[o] == y_test[o]).float().mean().item() + 0.5 * (pred[n] == y_test[n]).float().mean().item()

## 1. Grow the head ⏱ 8 min

`grow_head(old_model, n_total) -> new_model`: a fresh `Net(n_total)` (a separate object) with the old backbone
weights loaded and the old head rows copied. Its logits for the old classes must equal the old model's.

In [ ]:
grown = grow_head(deployed, 10)
assert grown is not deployed and grown.head.out_features == 10
grown.eval(); x = X_test[:16].to(device)
assert torch.allclose(grown(x)[:, :5], deployed(x), atol=1e-5)
print("section 1 ok")

## 2. Freeze the backbone ⏱ 4 min

`freeze_backbone(model)`: after the call, only `model.head` parameters have `requires_grad=True`.

In [ ]:
m = freeze_backbone(grow_head(deployed, 10))
trainable = {n for n, p in m.named_parameters() if p.requires_grad}
assert trainable == {"head.weight", "head.bias"}, trainable
print("section 2 ok")

## 3. Distillation loss ⏱ 6 min

`kd_loss(student_logits, teacher_logits, T) -> scalar`: Hinton KD, the KL divergence from the teacher's to the
student's temperature-`T` softmax, batch-averaged and multiplied by `T²`. Both inputs are `(B, C)`.

In [ ]:
s, t = torch.randn(6, 5), torch.randn(6, 5)
p_t, logp_s = F.softmax(t / 3, 1), F.log_softmax(s / 3, 1)
want = (p_t * (p_t.log() - logp_s)).sum(1).mean() * 9
assert torch.isclose(kd_loss(s, t, 3.0), want, atol=1e-5)
assert kd_loss(t, t, 3.0).abs() < 1e-6
print("section 3 ok")

## 4. LoRA from scratch ⏱ 12 min

`LoRALinear(base: nn.Linear, r, alpha)`: an `nn.Module` computing `base(x) + (alpha / r) · x Aᵀ Bᵀ`, with
`A: (r, in)` initialised randomly (small) and `B: (out, r)` initialised to **zeros**. The base layer is frozen, and
the only trainable parameters are `A` and `B`.

In [ ]:
base = nn.Linear(20, 7)
lora = LoRALinear(copy.deepcopy(base), r=4, alpha=8)
x = torch.randn(5, 20)
assert torch.allclose(lora(x), base(x)), "at init the LoRA layer must equal the base layer"
tp = [p for p in lora.parameters() if p.requires_grad]
assert sum(p.numel() for p in tp) == 4 * (20 + 7)
w0 = lora.base.weight.clone()
opt = torch.optim.SGD(tp, lr=0.1)
for _ in range(3):
    loss = lora(x).pow(2).mean(); opt.zero_grad(); loss.backward(); opt.step()
assert torch.equal(lora.base.weight, w0) and not torch.allclose(lora(x), base(x))
print("section 4 ok")

## 5. Class-incremental training ⏱ 15 min

`train_incremental(deployed, X_old, y_old, X_new, y_new) -> model`: return a 10-class model built from
`deployed` (don't retrain from scratch). The check needs `night_score(model) ≥ 0.72`
(½·acc_old + ½·acc_new on the full FashionMNIST test set).

In [ ]:
before = copy.deepcopy(deployed.state_dict())
model = train_incremental(deployed, X_old, y_old, X_new, y_new)
assert all(torch.equal(before[k], v) for k, v in deployed.state_dict().items()), "don't modify the deployed model"
s = night_score(model)
assert s >= 0.72, s
print(f"section 5 ok (score {s:.3f})")